In [ ]:
import pandas as pd
import numpy as np

DATASET_PATH = "sepsis_200_patient_subset(4).csv"

df = pd.read_csv('/content/sepsis_200_patient_subset.csv')

# -----------------------------
# Approximate SOFA Components
# -----------------------------

# Cardiovascular
df["SOFA_MAP"] = np.where(df["MAP"] < 70, 1, 0)

# Coagulation
df["SOFA_Platelets"] = np.select(
    [
        df["Platelets"] < 20,
        df["Platelets"] < 50,
        df["Platelets"] < 100,
        df["Platelets"] < 150
    ],
    [4, 3, 2, 1],
    default=0
)

# Liver
df["SOFA_Bilirubin"] = np.select(
    [
        df["Bilirubin_total"] >= 12,
        df["Bilirubin_total"] >= 6,
        df["Bilirubin_total"] >= 2
    ],
    [4, 3, 2],
    default=0
)

# Renal
df["SOFA_Creatinine"] = np.select(
    [
        df["Creatinine"] >= 5,
        df["Creatinine"] >= 3.5,
        df["Creatinine"] >= 2,
        df["Creatinine"] >= 1.2
    ],
    [4, 3, 2, 1],
    default=0
)

# Total Approximate SOFA
df["SOFA_Total"] = (
    df["SOFA_MAP"] +
    df["SOFA_Platelets"] +
    df["SOFA_Bilirubin"] +
    df["SOFA_Creatinine"]
)

# -----------------------------
# Modified Sepsis-3 Label
# -----------------------------

df["Sepsis3Label"] = (
    df["SOFA_Total"] >= 2
).astype(int)

print(df["Sepsis3Label"].value_counts())

df.to_csv(
    "sepsis3_dataset.csv",
    index=False
)

print("Saved: sepsis3_dataset.csv")

Sepsis3Label
0    141
1     59
Name: count, dtype: int64
Saved: sepsis3_dataset.csv


In [1]:
!pip -q install imbalanced-learn xgboost lightgbm pyarrow
%run psA_matched_run_native.py

Mounted at /content/drive
PREFLIGHT
  DATA_DIR -> /content
  OUT_DIR  -> /content/drive/MyDrive/psa_out
  input OK: features_native.parquet
  input OK: feature_tags.csv
  features: n=15309  prevalence=0.0736  sites={'A': np.int64(7821), 'B': np.int64(7488)}

  arm widths (expect 52|0/52, 130|130/0, 61|42/19):
    full                   total=243 | num=172 cat= 71
    lab_indicators_only    total=191 | num=172 cat= 19
    lab_values_dropped     total=113 | num= 42 cat= 71
    ordering_only          total= 52 | num=  0 cat= 52
    values_only            total=130 | num=130 cat=  0
    lab_panel              total= 61 | num= 42 cat= 19
  found prior psA_native_crosssite_preds.csv
    at   /content/psA_native_crosssite_preds.csv
    -> copied into OUT_DIR
  found prior psA_native_withinsrc_perfold.csv
    at   /content/psA_native_withinsrc_perfold.csv
    -> copied into OUT_DIR

  existing outputs:
    psA_native_crosssite_preds.csv      183,708 rows | arms=['full', 'lab_indicators_only'] 

Install required libraries

In [ ]:
!pip install -q imbalanced-learn xgboost

LOAD Dataset

In [ ]:
import pandas as pd
import numpy as np

DATASET_PATH = "sepsis3_dataset.csv"

df = pd.read_csv(DATASET_PATH)

print("Dataset loaded successfully")
print("Shape:", df.shape)
print("Columns:")
print(df.columns.tolist())
print(df.head())

Dataset loaded successfully
Shape: (200, 21)
Columns:
['PatientID', 'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'Resp', 'Creatinine', 'Bilirubin_total', 'Platelets', 'WBC', 'Lactate', 'Age', 'Gender', 'SepsisLabel', 'SOFA_MAP', 'SOFA_Platelets', 'SOFA_Bilirubin', 'SOFA_Creatinine', 'SOFA_Total', 'Sepsis3Label']
  PatientID     HR  O2Sat   Temp    SBP    MAP  Resp  Creatinine  \
0   p101527   71.0  100.0  36.60  128.0   57.0  22.0        3.02   
1   p014451   82.0   96.0  36.39  118.0   75.0  15.0        0.40   
2   p119433  107.0   98.0  36.00  158.0  123.0  22.0        1.56   
3   p013002  110.0   97.0  37.11  119.0   87.0  18.0        1.20   
4   p019421   81.0   99.0  37.94  165.0  107.0  13.0        1.00   

   Bilirubin_total  Platelets  ...  Lactate    Age  Gender  SepsisLabel  \
0              0.8       47.0  ...      NaN  69.00       0            1   
1              NaN      434.0  ...      0.9  61.14       1            1   
2              0.2      311.0  ...      NaN  68.00       0  

3: Dataset preparation

In [ ]:
if "SepsisLabel" not in df.columns and "Sepsis3Label" in df.columns:
    df = df.rename(columns={"Sepsis3Label": "SepsisLabel"})

if "SepsisLabel" not in df.columns:
    raise ValueError("Label column not found. Expected SepsisLabel or Sepsis2Label.")

df = df.copy()

print("Using label column: SepsisLabel")
print("Dataset shape:", df.shape)
print("Class distribution:")
print(df["SepsisLabel"].value_counts(dropna=False))

Using label column: SepsisLabel
Dataset shape: (200, 21)
Class distribution:
SepsisLabel
0    160
1     40
Name: count, dtype: int64


 4: Shared helper functions

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

all_results = []
all_feature_importance = []

def prepare_dataset(input_df):
    data = input_df.copy()

    if "SepsisLabel" not in data.columns and "Sepsis2Label" in data.columns:
        data = data.rename(columns={"Sepsis2Label": "SepsisLabel"})

    if set(["HR", "SBP"]).issubset(data.columns):
        data["Shock_Index"] = data["HR"] / (data["SBP"] + 1e-6)

    if set(["Creatinine", "Age"]).issubset(data.columns):
        data["Creatinine_Age"] = data["Creatinine"] * data["Age"]

    if set(["WBC", "Platelets"]).issubset(data.columns):
        data["WBC_Platelet_Ratio"] = data["WBC"] / (data["Platelets"] + 1e-6)

    if set(["Lactate", "MAP"]).issubset(data.columns):
        data["Lactate_MAP_Ratio"] = data["Lactate"] / (data["MAP"] + 1e-6)

    feature_names = [
        c for c in data.columns
        if c not in ["PatientID", "SepsisLabel", "Sepsis2Label"]
    ]

    X = data[feature_names]
    y = data["SepsisLabel"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    imputer = SimpleImputer(strategy="median")

    X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=feature_names)
    X_test = pd.DataFrame(imputer.transform(X_test), columns=feature_names)

    return X_train, X_test, y_train, y_test, feature_names

def get_models():
    return {
        "RandomForest": RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            random_state=42,
            n_jobs=-1
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=42,
            n_jobs=-1
        )
    }

def train_and_evaluate_sampling_model(sampling_name, sampler, result_csv, feature_csv):
    print(f"Starting {sampling_name}...")

    X_train, X_test, y_train, y_test, feature_names = prepare_dataset(df)

    if sampler is None:
        X_train_res, y_train_res = X_train, y_train
    else:
        X_train_res, y_train_res = sampler.fit_resample(X_train, y_train)

    print("Training shape:", X_train_res.shape)
    print("Class distribution after sampling:")
    print(pd.Series(y_train_res).value_counts())

    models = get_models()
    results = []
    feature_importance_all = []

    for name, model in models.items():
        print(f"Training {name}...")

        model.fit(X_train_res, y_train_res)

        pred = model.predict(X_test)
        prob = model.predict_proba(X_test)[:, 1]

        results.append({
            "Sampling": sampling_name,
            "Model": name,
            "CV_ROC_AUC": np.nan,
            "Accuracy": accuracy_score(y_test, pred),
            "Precision": precision_score(y_test, pred, zero_division=0),
            "Recall": recall_score(y_test, pred, zero_division=0),
            "F1": f1_score(y_test, pred, zero_division=0),
            "ROC_AUC": roc_auc_score(y_test, prob),
            "PR_AUC": average_precision_score(y_test, prob),
            "Brier_Score": brier_score_loss(y_test, prob)
        })

        if hasattr(model, "feature_importances_"):
            fi = pd.DataFrame({
                "Feature": feature_names,
                "Importance": model.feature_importances_,
                "Model": name,
                "Sampling": sampling_name
            })
            feature_importance_all.append(fi)

    results_df = pd.DataFrame(results)
    results_df.to_csv(result_csv, index=False)
    all_results.append(results_df)

    if feature_importance_all:
        feature_df = pd.concat(feature_importance_all).sort_values(
            "Importance",
            ascending=False
        )
        feature_df.to_csv(feature_csv, index=False)
        all_feature_importance.append(feature_df)

    print(f"Finished {sampling_name}")
    print(results_df)

    return results_df

MODEL 1: Baseline training, evaluation, metrics, save results

In [ ]:
baseline_results = train_and_evaluate_sampling_model(
    sampling_name="Baseline",
    sampler=None,
    result_csv="baseline_results.csv",
    feature_csv="baseline_feature_importance.csv"
)

Starting Baseline...
Training shape: (160, 23)
Class distribution after sampling:
SepsisLabel
0    128
1     32
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished Baseline
   Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall   F1  \
0  Baseline  RandomForest         NaN     0.850        1.0    0.25  0.4   
1  Baseline       XGBoost         NaN     0.775        0.0    0.00  0.0   

    ROC_AUC    PR_AUC  Brier_Score  
0  0.746094  0.601941     0.134346  
1  0.785156  0.449566     0.148206  


MODEL 2: SMOTE training, evaluation, metrics, save results

In [ ]:
smote_results = train_and_evaluate_sampling_model(
    sampling_name="SMOTE",
    sampler=SMOTE(random_state=42),
    result_csv="smote_results.csv",
    feature_csv="smote_feature_importance.csv"
)

Starting SMOTE...
Training shape: (256, 23)
Class distribution after sampling:
SepsisLabel
0    128
1    128
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished SMOTE
  Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall        F1  \
0    SMOTE  RandomForest         NaN     0.775   0.333333   0.125  0.181818   
1    SMOTE       XGBoost         NaN     0.825   0.666667   0.250  0.363636   

    ROC_AUC    PR_AUC  Brier_Score  
0  0.816406  0.456701     0.128672  
1  0.808594  0.468613     0.136086  


MODEL 3: ADASYN training, evaluation, metrics, save results

In [ ]:
adasyn_results = train_and_evaluate_sampling_model(
    sampling_name="ADASYN",
    sampler=ADASYN(random_state=42),
    result_csv="adasyn_results.csv",
    feature_csv="adasyn_feature_importance.csv"
)

Starting ADASYN...
Training shape: (256, 23)
Class distribution after sampling:
SepsisLabel
0    128
1    128
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished ADASYN
  Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall        F1  \
0   ADASYN  RandomForest         NaN       0.8        0.5   0.125  0.200000   
1   ADASYN       XGBoost         NaN       0.8        0.5   0.250  0.333333   

    ROC_AUC    PR_AUC  Brier_Score  
0  0.810547  0.504309     0.133401  
1  0.750000  0.419786     0.147395  


MODEL 4: BorderlineSMOTE training, evaluation, metrics, save results

In [ ]:
borderline_smote_results = train_and_evaluate_sampling_model(
    sampling_name="BorderlineSMOTE",
    sampler=BorderlineSMOTE(random_state=42),
    result_csv="borderline_smote_results.csv",
    feature_csv="borderline_smote_feature_importance.csv"
)

Starting BorderlineSMOTE...
Training shape: (256, 23)
Class distribution after sampling:
SepsisLabel
0    128
1    128
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished BorderlineSMOTE
          Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall  \
0  BorderlineSMOTE  RandomForest         NaN       0.8        0.5   0.125   
1  BorderlineSMOTE       XGBoost         NaN       0.8        0.5   0.250   

         F1   ROC_AUC    PR_AUC  Brier_Score  
0  0.200000  0.845703  0.505952     0.129318  
1  0.333333  0.820312  0.495234     0.130636  


MODEL 5: RandomUnderSampling training, evaluation, metrics, save results

In [ ]:
random_under_results = train_and_evaluate_sampling_model(
    sampling_name="RandomUnderSampling",
    sampler=RandomUnderSampler(random_state=42),
    result_csv="random_undersampling_results.csv",
    feature_csv="random_under_feature_importance.csv"
)

Starting RandomUnderSampling...
Training shape: (64, 23)
Class distribution after sampling:
SepsisLabel
0    32
1    32
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished RandomUnderSampling
              Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall  \
0  RandomUnderSampling  RandomForest         NaN       0.7   0.357143   0.625   
1  RandomUnderSampling       XGBoost         NaN       0.7   0.333333   0.500   

         F1   ROC_AUC    PR_AUC  Brier_Score  
0  0.454545  0.728516  0.473711     0.187660  
1  0.400000  0.738281  0.569635     0.172264  


MODEL 6: SMOTETomek training, evaluation, metrics, save results

In [ ]:
smote_tomek_results = train_and_evaluate_sampling_model(
    sampling_name="SMOTETomek",
    sampler=SMOTETomek(random_state=42),
    result_csv="smote_tomek_results.csv",
    feature_csv="smote_tomek_feature_importance.csv"
)

Starting SMOTETomek...
Training shape: (246, 23)
Class distribution after sampling:
SepsisLabel
0    123
1    123
Name: count, dtype: int64
Training RandomForest...
Training XGBoost...
Finished SMOTETomek
     Sampling         Model  CV_ROC_AUC  Accuracy  Precision  Recall  \
0  SMOTETomek  RandomForest         NaN       0.8        0.5    0.25   
1  SMOTETomek       XGBoost         NaN       0.8        0.5    0.25   

         F1   ROC_AUC    PR_AUC  Brier_Score  
0  0.333333  0.820312  0.480378     0.132882  
1  0.333333  0.769531  0.429749     0.148581  


FINAL CELL: Combine results, rank by ROC-AUC, export CSV

In [ ]:
comparison_df = pd.concat(all_results, ignore_index=True)

comparison_df = comparison_df.sort_values(
    by="ROC_AUC",
    ascending=False
).reset_index(drop=True)

comparison_df.insert(0, "Rank", range(1, len(comparison_df) + 1))

comparison_df.to_csv("model_comparison_by_roc_auc.csv", index=False)

print("Final model comparison ranked by ROC-AUC:")
print(comparison_df)

if all_feature_importance:
    combined_feature_importance = pd.concat(all_feature_importance, ignore_index=True)
    combined_feature_importance.to_csv("all_feature_importance.csv", index=False)
    print("Saved all_feature_importance.csv")

print("Saved model_comparison_by_roc_auc.csv")

Final model comparison ranked by ROC-AUC:
    Rank             Sampling         Model  CV_ROC_AUC  Accuracy  Precision  \
0      1      BorderlineSMOTE  RandomForest         NaN     0.800   0.500000   
1      2      BorderlineSMOTE       XGBoost         NaN     0.800   0.500000   
2      3           SMOTETomek  RandomForest         NaN     0.800   0.500000   
3      4                SMOTE  RandomForest         NaN     0.775   0.333333   
4      5               ADASYN  RandomForest         NaN     0.800   0.500000   
5      6                SMOTE       XGBoost         NaN     0.825   0.666667   
6      7             Baseline       XGBoost         NaN     0.775   0.000000   
7      8           SMOTETomek       XGBoost         NaN     0.800   0.500000   
8      9               ADASYN       XGBoost         NaN     0.800   0.500000   
9     10             Baseline  RandomForest         NaN     0.850   1.000000   
10    11  RandomUnderSampling       XGBoost         NaN     0.700   0.333333  